<a href="https://colab.research.google.com/github/FPetterini/labor/blob/main/MPL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo de Probabilidade Linear

**Econometria I · Capítulo 15**

**Pergunta da aula:** como o desempenho anterior e a participação em um método de ensino se relacionam com a probabilidade de um estudante obter um conceito elevado?

Vamos partir de uma regressão que já conhecemos, por mínimos quadrados ordinários (MQO), e mudar a natureza da variável dependente: ela terá apenas os valores **0 e 1**. O objetivo é compreender o significado dessa mudança, interpretar os coeficientes e comparar a inferência com matriz de variância-covariância clássica, robusta e estimada por bootstrap.

**Referência:** Gujarati e Porter, *Econometria básica*, 5ª edição, capítulo 15. Fundamentos: seções 15.1–15.4. Aplicação: dados da Tabela 15.7 (p. 559) e MQO da Tabela 15.14 (p. 567). Manteremos os dados originais do livro.

In [ ]:
# Bibliotecas: tabelas, cálculos, gráficos e estimação econométrica.

# Verifica se uma biblioteca está instalada no ambiente.
import importlib.util

# Executa comandos externos ao Python, como a instalação de bibliotecas.
import subprocess

# Permite identificar o Python usado neste ambiente.
import sys

# Se statsmodels não estiver instalada, instala usando o Python deste ambiente.
if importlib.util.find_spec("statsmodels") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])

# Permite ler um texto como se fosse um arquivo, útil para importar dados escritos no código.
from io import StringIO

# NumPy: realiza cálculos numéricos e operações com vetores e matrizes.
import numpy as np

# pandas: organiza e manipula dados em tabelas chamadas DataFrames.
import pandas as pd

# Matplotlib: constrói gráficos, como histogramas e diagramas de dispersão.
import matplotlib.pyplot as plt

# statsmodels: estima modelos econométricos e realiza testes estatísticos.
import statsmodels.api as sm

# display exibe tabelas e outros resultados; HTML permite formatar sua apresentação no notebook.
from IPython.display import display, HTML

# Faz os gráficos aparecerem no próprio notebook.
%matplotlib inline

# Define o tamanho das fontes e remove as bordas superior e direita dos gráficos.
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

# Limita a 12 o número de colunas exibidas nas tabelas do pandas.
pd.set_option("display.max_columns", 12)

print("Ambiente pronto. Os dados são dos 32 estudantes da Tabela 15.7.")

## 1. Entendendo o exemplo do livro no nosso contexto

O livro utiliza um estudo com **32 estudantes de um curso de Economia nos Estados Unidos**, de Spector e Mazzeo (1980). A pergunta é se características anteriores do aluno e um método de ensino ajudam a explicar seu desempenho final.

Nos Estados Unidos, é comum expressar o desempenho por **conceitos em letras**. No exemplo do livro, A indica um desempenho superior aos conceitos B e C. O estudo reúne B e C em uma categoria e A em outra.

| Variável | O que significa no estudo | Como pensar no contexto brasileiro |
|---|---|---|
| `Y` | 1 se o conceito final foi A; 0 se foi B ou C | Indicador de desempenho elevado. Como analogia, poderíamos criar 1 para nota final de pelo menos 9 e 0 para nota inferior a 9. Esse corte é apenas ilustrativo. |
| `GPA` (*Grade Point Average*) | Média de desempenho acadêmico anterior, na escala norte-americana de pontos, usualmente de 0 a 4 | Tem função semelhante à de um índice de aproveitamento ou coeficiente de rendimento. **GPA = 3,2 não é uma nota 3,2 na escala brasileira de 0 a 10.** |
| `TUCE` (*Test of Understanding in College Economics*) | Pontuação em um teste de conhecimentos de Economia aplicado no início do semestre | Pense em uma prova de nivelamento ou pré-teste, que mede a preparação inicial. Seus pontos não são notas de 0 a 10. |
| `PSI` (*Personalized System of Instruction*) | 1 se o estudante participou do método de ensino alternativo; 0 se não participou | Pense em um indicador que distingue o método novo do método habitual de uma disciplina. É uma categoria, e não uma nota. |

**As analogias ajudam a entender as variáveis; não estamos convertendo as notas nem criando uma base brasileira.** A regressão usará os valores originais.

Observe também: **`Y = 0` não significa nota zero nem reprovação**. Significa somente que o conceito foi B ou C, e não A. Já `PSI = 0` significa ausência de participação no método alternativo. As duas variáveis usam 0 e 1, mas representam coisas diferentes.

**Pergunta inicial:** seria razoável esperar que alunos com melhor desempenho anterior tivessem maior probabilidade de obter A?

In [ ]:
# Valores da Tabela 15.7. Cada linha representa um estudante.
csv_estudantes = """GPA,TUCE,PSI,Y
2.66,20,0,0
2.89,22,0,0
3.28,24,0,0
2.92,12,0,0
4.00,21,0,1
2.86,17,0,0
2.76,17,0,0
2.87,21,0,0
3.03,25,0,0
3.92,29,0,1
2.63,20,0,0
3.32,23,0,0
3.57,23,0,0
3.26,25,0,1
3.53,26,0,0
2.74,19,0,0
2.75,25,0,0
2.83,19,0,0
3.12,23,1,0
3.16,25,1,1
2.06,22,1,0
3.62,28,1,1
2.89,14,1,0
3.51,26,1,0
3.54,24,1,1
2.83,27,1,1
3.39,17,1,1
2.67,24,1,0
3.65,21,1,1
4.00,23,1,1
3.10,21,1,0
2.39,19,1,1
"""

dados = pd.read_csv(StringIO(csv_estudantes))
dados.index = pd.RangeIndex(1, len(dados) + 1, name="Estudante")
print(f"Número de estudantes: {len(dados)}")
display(dados.head(6))

## 2. Por que o nome Modelo de Probabilidade Linear?

### Passo 1. A variável dependente é binária

Em uma regressão usual de notas, a variável dependente poderia assumir vários valores: 5,0; 6,5; 8,0; 9,2 etc. Aqui decidimos estudar um **evento**: obter conceito A. Registramos apenas se o evento aconteceu:

$$
Y_i=\begin{cases}
1, & \text{se o estudante obteve A;}\\
0, & \text{se obteve B ou C.}
\end{cases}
$$

Uma variável com esses dois resultados possíveis é chamada **binária** ou **dicotômica**. O foco desta aula é a variável **dependente** binária; a presença de uma dummy entre os regressores, por si só, não define um MPL.

### Passo 2. A média dos zeros e uns é a proporção de ocorrências

Imagine, por um momento, 10 alunos: 4 receberam o código 1 e 6 receberam o código 0. A média será:

$$\overline Y=\frac{4\times 1+6\times 0}{10}=0{,}4.$$

Esse 0,4 significa que **40% dos alunos obtiveram o resultado de interesse**. Não significa que alguém recebeu uma nota 0,4. Nos dados do livro, a mesma conta é 11/32, aproximadamente 34,38%.

In [ ]:
print(f"Estudantes com conceito A: {dados['Y'].sum()}")
print(f"Proporção com conceito A: {dados['Y'].mean():.2%}")

# Comparação descritiva entre os dois grupos de ensino.
por_metodo = dados.groupby("PSI").agg(
    estudantes=("Y", "size"),
    proporcao_A=("Y", "mean"),
    GPA_medio=("GPA", "mean"),
    TUCE_medio=("TUCE", "mean"),
)
display(por_metodo.style.format({
    "proporcao_A": "{:.2%}", "GPA_medio": "{:.2f}", "TUCE_medio": "{:.2f}"
}))

In [ ]:
# Funções para calcular os intervalos de confiança.
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import t

# IC de 95% para uma proporção, considerando que Y assume valores 0 ou 1.
def ic_proporcao(serie):
    valores = serie.dropna()
    inferior, superior = proportion_confint(
        count=valores.sum(),  # Número de estudantes com conceito A.
        nobs=len(valores),    # Número de estudantes com Y observado.
        alpha=0.05,          # Nível de confiança: 1 - 0,05 = 95%.
        method="wilson"
    )
    return f"[{inferior:.2%}; {superior:.2%}]"

# IC de 95% para uma média: média ± valor crítico t × erro-padrão.
def ic_media(serie):
    valores = serie.dropna()
    n = len(valores)
    media = valores.mean()
    erro_padrao = valores.std(ddof=1) / np.sqrt(n)
    margem = t.ppf(0.975, df=n - 1) * erro_padrao

    return f"[{media - margem:.2f}; {media + margem:.2f}]"

# Resultados para o conjunto dos estudantes.
print(f"Estudantes com conceito A: {dados['Y'].sum()}")
print(f"Proporção com conceito A: {dados['Y'].mean():.2%}")
print(f"IC de 95% para essa proporção: {ic_proporcao(dados['Y'])}")

# Comparação entre os dois grupos de ensino.
# Cada intervalo aparece ao lado da respectiva proporção ou média.
por_metodo = dados.groupby("PSI").agg(
    estudantes=("Y", "size"),
    proporcao_A=("Y", "mean"),
    IC95_proporcao_A=("Y", ic_proporcao),
    GPA_medio=("GPA", "mean"),
    IC95_GPA=("GPA", ic_media),
    TUCE_medio=("TUCE", "mean"),
    IC95_TUCE=("TUCE", ic_media),
)

display(por_metodo.style.format({
    "proporcao_A": "{:.2%}",
    "GPA_medio": "{:.2f}",
    "TUCE_medio": "{:.2f}"
}))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import t

# Cada grupo ocupa uma linha do gráfico.
grupos = [0, 1]
rotulos = ["Ensino tradicional (PSI = 0)", "Método PSI (PSI = 1)"]
cores = ["#2563eb", "#d97706"]

# Um gráfico para cada variável.
variaveis = {
    "Y": "Proporção de estudantes com conceito A",
    "GPA": "Média do GPA",
    "TUCE": "Média do TUCE",
}

for variavel, titulo in variaveis.items():
    fig, ax = plt.subplots(figsize=(9, 3.2))

    for posicao, grupo in enumerate(grupos):

        # Seleciona os valores da variável no grupo.
        valores = dados.loc[dados["PSI"] == grupo, variavel].dropna()
        media = valores.mean()

        if variavel == "Y":
            # Para Y binária, calcula o IC da proporção pelo método de Wilson.
            inferior, superior = proportion_confint(
                count=valores.sum(),
                nobs=len(valores),
                alpha=0.05,
                method="wilson"
            )
        else:
            # Para GPA e TUCE, calcula o IC da média usando t de Student.
            erro_padrao = valores.sem()
            margem = t.ppf(0.975, df=len(valores) - 1) * erro_padrao
            inferior = media - margem
            superior = media + margem

        # Linha de referência horizontal, semelhante à imagem.
        ax.axhline(posicao, color="0.88", linewidth=1, zorder=0)

        # Ponto = média; barra horizontal = intervalo de confiança.
        # As distâncias são calculadas separadamente porque o IC de
        # Wilson pode ser assimétrico em relação à proporção observada.
        ax.errorbar(
            x=media,
            y=posicao,
            xerr=[[media - inferior], [superior - media]],
            fmt="o",
            markersize=8,
            capsize=7,
            elinewidth=2,
            color=cores[posicao],
            zorder=3
        )

        # Exibe o valor da média acima do ponto.
        texto = f"{media:.1%}" if variavel == "Y" else f"{media:.2f}"
        ax.annotate(
            texto,
            xy=(media, posicao),
            xytext=(0, 12),
            textcoords="offset points",
            ha="center",
            color=cores[posicao]
        )

    # Identifica os grupos e coloca o grupo tradicional na primeira linha.
    ax.set_yticks([0, 1])
    ax.set_yticklabels(rotulos)
    ax.set_ylim(1.5, -0.5)

    ax.set_title(f"{titulo} — IC de 95%", pad=15)
    ax.set_xlabel("Proporção" if variavel == "Y" else variavel)
    ax.tick_params(axis="y", length=0)
    ax.margins(x=0.15)

    # Na variável Y, apresenta a escala em porcentagem.
    if variavel == "Y":
        ax.set_xlim(0, 1)
        ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))

    for borda in ["top", "right", "left"]:
        ax.spines[borda].set_visible(False)

    plt.tight_layout()
    plt.show()

### Passo 3. A esperança matemática também é uma média, ponderada por probabilidades

A esperança de uma variável discreta é a soma de cada resultado possível multiplicado por sua probabilidade. Se $p=P(Y=1)$, então:

| Resultado possível | Probabilidade | Contribuição para a esperança |
|---|---|---|
| $Y=1$ | $p$ | $1\times p$ |
| $Y=0$ | $1-p$ | $0\times(1-p)$ |

Somando as contribuições:

$$E(Y)=1\times p+0\times(1-p)=p.$$

Por exemplo, se a probabilidade de obter A for 40%, a esperança dessa variável binária será 0,4, embora cada realização continue sendo 0 ou 1. A **média amostral** é calculada nos dados; a **esperança** é uma característica da distribuição probabilística.

### Passo 4. Na regressão, queremos a probabilidade para um determinado perfil

Alunos com preparação inicial diferente podem ter probabilidades diferentes. Escrevemos $X_i$ para representar os regressores do estudante: GPA, TUCE e PSI. Condicionar em $X_i$ significa considerar esse perfil:

$$
E(Y_i\mid X_i)=1\times P(Y_i=1\mid X_i)+0\times P(Y_i=0\mid X_i)
=P(Y_i=1\mid X_i)=p_i.
$$

**É por isso que uma regressão para a média condicional de uma variável 0/1 pode ser interpretada como uma regressão para uma probabilidade.**

### Passo 5. Chamamos o modelo de linear porque adotamos uma soma linear

Vamos representar essa probabilidade por:

$$p_i=\beta_1+\beta_2 GPA_i+\beta_3 TUCE_i+\beta_4 PSI_i.$$

Assim, o nome reúne duas ideias: **probabilidade**, porque a média condicional de $Y$ binária é $P(Y=1\mid X)$; **linear**, porque usamos essa forma linear nos parâmetros. Nesta especificação, sem interações ou termos adicionais, a mudança prevista por unidade de cada regressor é constante.

**Confira se entendeu:** por que uma previsão de 0,6 pode fazer sentido, se nenhum estudante tem `Y = 0,6`?

## 3. Estimando o MPL e interpretando os resultados

O modelo de regressão é:

$$Y_i=\beta_1+\beta_2 GPA_i+\beta_3 TUCE_i+\beta_4 PSI_i+u_i.$$

Usaremos MQO, como nas regressões anteriores. O comando `add_constant` acrescenta o intercepto; `OLS` define a regressão; `fit()` estima os coeficientes. A tabela inicial usa a matriz de variância-covariância clássica, para reproduzir a referência do livro. Depois compararemos a inferência.

In [ ]:
regressores = ["GPA", "TUCE", "PSI"]
X = sm.add_constant(dados[regressores], has_constant="add")
y = dados["Y"]
mpl = sm.OLS(y, X).fit()

tabela_mqo = pd.DataFrame({
    "Coeficiente": mpl.params,
    "EP clássico": mpl.bse,
    "t clássico (H0: coef. = 0)": mpl.tvalues,
    "p-valor clássico": mpl.pvalues,
})
display(tabela_mqo.style.format("{:.4f}"))
print(f"Observações: {int(mpl.nobs)} | R²: {mpl.rsquared:.4f}")

### Os coeficientes medem mudanças na probabilidade

Os resultados reproduzem, por arredondamento, a Tabela 15.14:

$$\widehat p_i\approx-1{,}4980+0{,}4639\,GPA_i+0{,}0105\,TUCE_i+0{,}3786\,PSI_i.$$

Mantendo os demais regressores constantes:

- Um aumento de **0,1 ponto no GPA**, na escala original, está associado a aproximadamente **4,64 pontos percentuais** a mais na probabilidade prevista de obter A.
- Um ponto adicional no **TUCE** está associado a aproximadamente **1,05 ponto percentual** a mais.
- Passar de **PSI = 0 para PSI = 1** está associado a uma diferença ajustada de aproximadamente **37,86 pontos percentuais**.

Pontos percentuais são diferenças entre probabilidades expressas em %. Passar de 20% para 30% representa **10 pontos percentuais**. O intercepto corresponde a um perfil com GPA e TUCE iguais a zero, fora do suporte desta base; não oferece uma interpretação substantiva útil aqui.

Essas são associações ajustadas. Interpretá-las causalmente requer justificar os pressupostos sobre a relação entre os regressores e os fatores não observados.

In [ ]:
# Exemplo resolvido: um aumento de 0,1 ponto no GPA.
delta_gpa_exemplo = 0.1
delta_p_exemplo = mpl.params["GPA"] * delta_gpa_exemplo
print(f"Aumento de {delta_gpa_exemplo} no GPA: {100 * delta_p_exemplo:.2f} pontos percentuais.")
print(f"Mudança de PSI = 0 para PSI = 1: {100 * mpl.params['PSI']:.2f} pontos percentuais.")

### Previsões para dois perfis

Os perfis abaixo têm GPA = 3,2 e TUCE = 22, variando apenas PSI. Uma previsão de 0,60, por exemplo, significa uma probabilidade prevista de 60% para aquele perfil, e não uma nota 0,60.

In [ ]:
perfis_guiados = pd.DataFrame({
    "GPA": [3.2, 3.2],
    "TUCE": [22, 22],
    "PSI": [0, 1],
}, index=["Sem PSI", "Com PSI"])

X_perfis = sm.add_constant(perfis_guiados[regressores], has_constant="add")
perfis_guiados["p_mpl"] = mpl.predict(X_perfis)
display(perfis_guiados.style.format({"GPA": "{:.1f}", "p_mpl": "{:.2%}"}))

diferenca_guiada = perfis_guiados.loc["Com PSI", "p_mpl"] - perfis_guiados.loc["Sem PSI", "p_mpl"]
print(f"Diferença prevista: {100 * diferenca_guiada:.2f} pontos percentuais.")

### Um limite importante da forma linear

A probabilidade verdadeira deve estar entre 0 e 1. Como MQO não impõe essa restrição, alguns valores ajustados podem sair desse intervalo. Vamos observar o que ocorre na própria amostra.

In [ ]:
dados["p_mpl"] = mpl.predict(X)
fora_da_faixa = ~dados["p_mpl"].between(0, 1)

print(f"Menor previsão: {dados['p_mpl'].min():.4f}")
print(f"Maior previsão: {dados['p_mpl'].max():.4f}")
print(f"Previsões abaixo de 0: {(dados['p_mpl'] < 0).sum()}")
print(f"Previsões acima de 1: {(dados['p_mpl'] > 1).sum()}")
display(dados.loc[fora_da_faixa, ["GPA", "TUCE", "PSI", "Y", "p_mpl"]].round(4))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))

# Painel 1: resultados observados e valores ajustados dos 32 estudantes.
ax = eixos[0]
ax.axhspan(0, 1, color="#edf5f7", zorder=0)
ax.scatter(dados.index, dados["Y"], color="#6b7280", s=28, label="Y observado (0 ou 1)")
ax.scatter(dados.index, dados["p_mpl"], color="#174a70", marker="x", s=48, label="Previsão do MPL")
ax.scatter(dados.index[fora_da_faixa], dados.loc[fora_da_faixa, "p_mpl"],
           facecolors="none", edgecolors="#bc3d38", s=110, linewidths=1.5,
           label="Fora de [0, 1]")
ax.set(xlabel="Estudante", ylabel="Resultado / previsão", title="O MPL pode gerar valores negativos")
ax.legend(fontsize=8, loc="lower left")

# Painel 2: perfis hipotéticos, mantendo TUCE na média amostral.
ax = eixos[1]
ax.axhspan(0, 1, color="#edf5f7", zorder=0)
grade_gpa = np.linspace(dados["GPA"].min(), dados["GPA"].max(), 100)
for psi, cor in [(0, "#174a70"), (1, "#18806c")]:
    grade = pd.DataFrame({"GPA": grade_gpa, "TUCE": dados["TUCE"].mean(), "PSI": psi})
    grade_X = sm.add_constant(grade[regressores], has_constant="add")
    ax.plot(grade_gpa, mpl.predict(grade_X), color=cor, linewidth=2.3, label=f"PSI = {psi}")
ax.set(xlabel="GPA", ylabel="Previsão do MPL", title="Efeito constante e retas paralelas")
ax.text(0.03, 0.94, f"TUCE fixado em {dados['TUCE'].mean():.2f}", transform=ax.transAxes, fontsize=9)
ax.legend(fontsize=9, loc="lower right")

for ax in eixos:
    ax.axhline(0, color="#8c929b", linestyle="--", linewidth=0.8)
    ax.axhline(1, color="#8c929b", linestyle="--", linewidth=0.8)
    ax.set_ylim(-0.4, 1.15)
    ax.grid(alpha=0.15)
fig.tight_layout()
plt.show()

Nesta amostra há **cinco previsões negativas** e nenhuma acima de 1. No segundo gráfico, o efeito do GPA e a diferença associada ao PSI são constantes. Guarde essas limitações da forma linear para a próxima aula. Alterar a matriz de variância-covariância não modifica esses valores ajustados.

## 4. Exogeneidade, viés, consistência e variância

### 4.1. A variável dependente binária torna MQO viesado?

**O fato de $Y$ assumir apenas 0 e 1 não produz, por si só, viés ou inconsistência em MQO.** Precisamos separar os pressupostos sobre a média do erro dos pressupostos sobre sua variância.

Nesta discussão, adotamos a hipótese de exogeneidade na forma:

$$E(u_i\mid X_i)=0.$$

Ela significa que, para cada perfil de regressores, os fatores que ficaram no erro têm média zero. Por exemplo, se uma motivação não observada estiver no erro e se relacionar sistematicamente com PSI mesmo após os controles, essa hipótese pode falhar.

Sob essa hipótese, com a média condicional corretamente especificada e sem colinearidade perfeita, os coeficientes de MQO são **não viesados**. Com condições usuais de amostragem e momentos, também são **consistentes**:

| Propriedade | Ideia em palavras |
|---|---|
| Ausência de viés | Se repetíssemos a coleta muitas vezes, a média das estimativas coincidiria com o parâmetro. |
| Consistência | Conforme a amostra cresce, a estimativa se aproxima do parâmetro. |

Essas propriedades não exigem homocedasticidade nem normalidade do erro. A hipótese de que uma reta representa corretamente a média condicional também importa: **ausência de endogeneidade não garante, sozinha, que a forma linear seja adequada**.

### 4.2. Então, o que muda por $Y$ ser binária?

Seja $p_i=P(Y_i=1\mid X_i)$, a verdadeira probabilidade condicional. Uma variável Bernoulli tem variância:

$$\operatorname{Var}(Y_i\mid X_i)=p_i(1-p_i).$$

Mantendo o perfil fixo, a parte explicada é um número. Subtraí-la de $Y_i$ não altera a variância; no MPL corretamente especificado, $u_i=Y_i-p_i$. Portanto:

$$\operatorname{Var}(u_i\mid X_i)=p_i(1-p_i).$$

Se a probabilidade varia entre os perfis, essa variância geralmente também varia: temos **heterocedasticidade**. Para um mesmo perfil, o erro só pode ser $1-p_i$ ou $-p_i$, de modo que a normalidade condicional do erro não se aplica.

In [ ]:
# Exemplo teórico: probabilidades diferentes implicam dispersões diferentes.
exemplo_variancia = pd.DataFrame({"Probabilidade p": [0.1, 0.5, 0.9]})
exemplo_variancia["Variância p(1-p)"] = (
    exemplo_variancia["Probabilidade p"] * (1 - exemplo_variancia["Probabilidade p"])
)
display(exemplo_variancia.style.format("{:.2f}"))

### 4.3. Coeficiente, desvio-padrão residual e erro padrão são coisas diferentes

| Quantidade | O que informa |
|---|---|
| Coeficiente estimado | A associação estimada entre o regressor e a probabilidade. |
| Desvio-padrão residual | Uma medida global de dispersão dos resíduos. Não representa uma variância condicional igual para todos os perfis. |
| Erro padrão do coeficiente | A incerteza amostral da estimativa daquele coeficiente. É obtido da matriz de variância-covariância dos estimadores. |

Na matriz de variância-covariância, cada elemento da **diagonal** é uma variância de um coeficiente estimado. A raiz quadrada dessa variância é seu **erro padrão**. Os elementos fora da diagonal são covariâncias entre estimadores.

Para testar $H_0:\beta_j=0$, calculamos a razão:

$$t_j=\frac{\widehat\beta_j}{EP(\widehat\beta_j)}.$$

Se a matriz clássica estima incorretamente a incerteza, os erros padrão, as estatísticas t, os p-valores e os intervalos de confiança podem ficar inadequados. **Podemos ter coeficientes não viesados e consistentes, mas inferência clássica inadequada.**

<details>
<summary><strong>Detalhamento matemático para acompanhar com o professor</strong></summary>

Sob exogeneidade, $\widehat\beta=\beta+(X'X)^{-1}X'u$, de modo que $E(\widehat\beta\mid X)=\beta$. Sob condições usuais, $X'u/n\to_p0$ e $X'X/n\to_p Q$, com $Q$ invertível, implicam consistência.

Para observações independentes, a variância condicional correta é:

$$\operatorname{Var}(\widehat\beta\mid X)=(X'X)^{-1}X'\Omega X(X'X)^{-1},\qquad
\Omega=\operatorname{diag}\{p_i(1-p_i)\}.$$

A expressão clássica $s^2(X'X)^{-1}$ usa uma variância comum. Já HC1 estima a matriz sem impor essa homocedasticidade:

$$\widehat V_{HC1}=\frac{n}{n-k}(X'X)^{-1}X'\operatorname{diag}(\widehat u_i^2)X(X'X)^{-1},$$

com $k=4$ parâmetros nesta aplicação, incluindo o intercepto.

</details>

## 5. Comparando matriz clássica, HC1 e bootstrap

### 5.1. Correção robusta disponível no pacote

O Colab executa o código Python. O pacote **statsmodels** oferece as correções HC0, HC1, HC2 e HC3. Usaremos **HC1**, uma correção robusta à heterocedasticidade, com ajuste por graus de liberdade, em linha com a discussão de White no capítulo.

O comando abaixo conserva a estimação por MQO e calcula outra matriz. Observe o que ocorre com coeficientes, erros padrão e estatísticas t.

In [ ]:
mpl_robusto = mpl.get_robustcov_results(cov_type="HC1", use_t=True)
nomes = list(mpl.params.index)

def tabela_inferencia(resultado):
    return pd.DataFrame({
        "Coeficiente": np.asarray(resultado.params),
        "Erro padrão": np.asarray(resultado.bse),
        "t (H0: coef. = 0)": np.asarray(resultado.tvalues),
        "p-valor": np.asarray(resultado.pvalues),
    }, index=nomes)

comparacao_inferencia = pd.concat({
    "Clássica": tabela_inferencia(mpl),
    "Robusta HC1": tabela_inferencia(mpl_robusto),
}, names=["Matriz", "Parâmetro"])
display(comparacao_inferencia.style.format("{:.4f}"))
print(f"Desvio-padrão residual global, s = raiz(SQR/(n-k)): {np.sqrt(mpl.scale):.4f}")
print("Os coeficientes, os resíduos, as previsões e o R² permanecem os mesmos.")

**Leia primeiro a linha de PSI.** O coeficiente continua aproximadamente 0,3786. O erro padrão passa de aproximadamente 0,1392 para 0,1504; por isso, a estatística t cai de aproximadamente 2,7200 para 2,5176. No GPA, o erro padrão robusto fica menor. Não existe uma regra segundo a qual todo erro padrão robusto deva aumentar.

Com `use_t=True`, escolhemos explicitamente a distribuição t como referência para os p-valores. Sob heterocedasticidade e erro não normal, essa inferência é aproximada; a correção robusta não torna o teste exato com 32 observações.

### 5.2. Bootstrap: estimar a incerteza por reamostragem

Outra maneira de obter erros padrão é observar como as estimativas variam em muitas reamostras. Aqui faremos **bootstrap de pares**, apropriado à demonstração com observações independentes:

1. Sortear 32 linhas da base **com reposição**: um estudante pode aparecer mais de uma vez, e outro pode não aparecer.
2. Ao sortear uma linha, levar juntos seu `Y`, `GPA`, `TUCE` e `PSI`.
3. Estimar MQO nessa reamostra e guardar os quatro coeficientes.
4. Repetir o procedimento 2.000 vezes. O desvio-padrão de cada coluna de coeficientes é seu erro padrão bootstrap; a covariância entre as colunas forma a matriz bootstrap.

O código está pronto. A semente fixa torna a demonstração reproduzível. Não estamos impondo uma variância residual comum ao reamostrar as linhas completas.

In [ ]:
B = 2000
rng = np.random.default_rng(20261031)
X_array = X.to_numpy()
y_array = y.to_numpy()
n, k = X_array.shape
coeficientes_boot = []

for _ in range(B):
    indices = rng.integers(0, n, size=n)  # sorteio com reposição
    X_b, y_b = X_array[indices], y_array[indices]
    if np.linalg.matrix_rank(X_b) < k:
        continue  # não estimar uma reamostra sem identificação dos parâmetros
    coeficientes_boot.append(sm.OLS(y_b, X_b).fit().params)

coeficientes_boot = np.asarray(coeficientes_boot)
if len(coeficientes_boot) < 2:
    raise ValueError("São necessárias pelo menos duas reamostras válidas.")

V_bootstrap = np.cov(coeficientes_boot, rowvar=False, ddof=1)
ep_bootstrap = pd.Series(np.sqrt(np.diag(V_bootstrap)), index=nomes)
ic_bootstrap = np.quantile(coeficientes_boot, [0.025, 0.975], axis=0)

comparacao_ep = pd.DataFrame({
    "Coeficiente MQO original": mpl.params,
    "EP clássico": mpl.bse,
    "EP robusto HC1": mpl_robusto.bse,
    "EP bootstrap": ep_bootstrap,
})
display(comparacao_ep.style.format("{:.4f}"))
print(f"Reamostras válidas: {len(coeficientes_boot)} de {B}.")

resumo_bootstrap = pd.DataFrame({
    "Média nas reamostras": coeficientes_boot.mean(axis=0),
    "Limite inferior 95% (percentis)": ic_bootstrap[0],
    "Limite superior 95% (percentis)": ic_bootstrap[1],
}, index=nomes)
display(resumo_bootstrap.style.format("{:.4f}"))

O coeficiente que reportamos continua sendo o MQO da amostra original. A média das estimativas nas reamostras pode diferir dele; ela não substitui automaticamente a estimativa original.

Os limites mostrados são os percentis 2,5% e 97,5% das estimativas reamostradas, um intervalo bootstrap percentil ilustrativo. **Não estamos realizando um teste bootstrap-t.** Podemos calcular a razão estimativa/EP bootstrap para comparação, mas ela não adquire automaticamente uma distribuição t de Student. Por isso, não atribuímos um p-valor t a essa razão.

HC1 e bootstrap estimam a incerteza. Sua validade depende de condições de amostragem e regularidade; nenhum deles elimina endogeneidade nem corrige uma forma funcional inadequada.

In [ ]:
# Ilustração: dispersão das estimativas de PSI nas reamostras.
j_psi = nomes.index("PSI")
fig, eixos = plt.subplots(1, 2, figsize=(11.5, 3.8))

eixos[0].hist(coeficientes_boot[:, j_psi], bins=30, color="#174a70", alpha=0.85)
eixos[0].axvline(mpl.params["PSI"], color="#bc3d38", linewidth=2, label="MQO original")
eixos[0].set(xlabel="Coeficiente de PSI", ylabel="Número de reamostras",
             title="Bootstrap: a estimativa varia entre reamostras")
eixos[0].legend(fontsize=9)

ep_psi = [mpl.bse["PSI"], mpl_robusto.bse[j_psi], ep_bootstrap["PSI"]]
eixos[1].bar(["Clássico", "HC1", "Bootstrap"], ep_psi,
             color=["#7f8993", "#18806c", "#174a70"], width=0.65)
for i, valor in enumerate(ep_psi):
    eixos[1].text(i, valor + 0.003, f"{valor:.4f}", ha="center", fontsize=10)
eixos[1].set(ylabel="Erro padrão de PSI", title="Mesma estimativa original, diferentes EPs",
             ylim=(0, max(ep_psi) * 1.25))
fig.tight_layout()
plt.show()

### Conferindo a matriz e sua diagonal

Abra o quadro abaixo se quiser examinar os números das três matrizes. Para cada parâmetro, a raiz da diagonal corresponde ao erro padrão apresentado na comparação. A matriz bootstrap é calculada pela dispersão dos coeficientes nas reamostras.

In [ ]:
matrizes_cov = {
    "Clássica": pd.DataFrame(mpl.cov_params(), index=nomes, columns=nomes),
    "Robusta HC1": pd.DataFrame(mpl_robusto.cov_params(), index=nomes, columns=nomes),
    "Bootstrap": pd.DataFrame(V_bootstrap, index=nomes, columns=nomes),
}
quadro = "<details><summary><strong>Abrir matrizes de variância-covariância</strong></summary>"
for nome, matriz in matrizes_cov.items():
    quadro += f"<h4>{nome}</h4>" + matriz.to_html(float_format=lambda valor: f"{valor:.5f}")
quadro += "</details>"
display(HTML(quadro))